# **PyTorch建立神经网络模型，MNIST数据库训练模型**



**首先，用PyTorch建神经网络模型**

In [2]:
from torch import nn
from torchsummary import summary

class_names = range(10)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_stack = nn.Sequential(
            nn.Linear(28*28, 96),
            nn.Tanh(),
            nn.Dropout(.2),

            nn.Linear(96, 256),
            nn.Sigmoid(),
            nn.Dropout(.2),

            nn.Linear(256, len(class_names)),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

以上程序片段用PyTorch构建神经网络，首先定义一个继承自PyTorch通用神经网络模块**nn.Module**的Python子类，此处命名为**NeuralNetwork**，它包含两个主要组件。

**init()初始化方法**：

此方法用作类的构造函数。首先使用**super(NeuralNetwork, self).init()** 初始化 **nn.Module**，在此方法内，定义前馈神经网络的架构。使用 **nn.Flatten()** 将输入从原始的*28x28*像素格式展平为一个包含*784*个元素的一维数组。接下来，使用 **nn.Sequential** 创建一个顺序堆叠的层，该网络包含：

*   第一个是包含96个神经元的隐藏层，每个神经元具有Tanh激活函数
*   每次训练 **dropout** 20%神经元，以防止过拟合
*   第二个是包含256个神经元的隐藏层，每个神经元具有Sigmoid激活函数
*   同样每次训练 **dropout** 20%神经元，以防过拟合，提升泛化性能
*   最后一个输出线性层，包含10个神经元（与数据集中的类别数量一致），连接Softmax激活函数，输出类别概率。

**前向传播方法**

此方法定义了网络的前向传播过程。它接收一个输入图片x，使用已定义的类中二个方法 **self.flatten()** 将其展平，然后将其依次传递给定义的网络层 **self.linear_stack（）**。输出结果为*logits*，表示模型所预测数字的类别概率值。


下一步实例化模型，并使用 torchsummary 包显示神经网络结构。



In [3]:
model = NeuralNetwork()

summary(model, (1, 28, 28))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                   [-1, 96]          75,360
              Tanh-3                   [-1, 96]               0
           Dropout-4                   [-1, 96]               0
            Linear-5                  [-1, 256]          24,832
           Sigmoid-6                  [-1, 256]               0
           Dropout-7                  [-1, 256]               0
            Linear-8                   [-1, 10]           2,570
           Softmax-9                   [-1, 10]               0
Total params: 102,762
Trainable params: 102,762
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.39
Estimated Total Size (MB): 0.41
-------------------------------------------

输出结果和README文档中的分析一致。

您已成功使用 PyTorch 定义并初始化了一个前馈神经网络。该模型旨在对 **MNIST** 数据集中的手写数字进行分类，其架构详情已通过 *summary()* 函数打印出来。该网络包含一个输入展平层、两个带有激活函数和 *dropout* 调节的隐藏层，以及一个使用 **softmax** 函数的输出层，用于预测数字类别的概率。您还确认该模型共有 *102,762* 个可训练参数。

下一步是使用 **MNIST** 数据集训练模型，这包括将数据输入网络、计算损失，并基于反向传播优化权重，以提高模型在数字分类中的准确率。

**用MNIST数据库中的训练集训练模型**

用大量数据训练神经网络的目的，是通过神经网络作为黑盒子来拟合无法用严格数学公式刻画的，产生大量数据的内部规律，让大量神经网络参数集合来描述大量数据所隐形拥有的内在规则。

在 PyTorch 中训练神经网络的典型方法包含以下几个关键步骤：

• 预处理数据集，例如对数据进行归一化并将其转换为合适的格式

• 将数据集划分为训练集和测试集，使用训练数据来更新模型的参数，并使用测试数据来评估其性能

• 将数据分批输入到网络中

• 使用损失函数计算预测误差或损失，对于分类任务，可以使用交叉熵损失函数

• 使用反向传播算法优化模型的权重，以逐步减小预测误差。反向传播算法计算损失函数相对于每个参数的梯度，然后使用优化器（例如随机梯度下降SGD或Adam）更新参数。

• 重复上述过程多个epoch，直到模型达到令人满意的性能，并在准确率和泛化能力之间取得平衡。

**训练模型时的关键因素**

在 PyTorch 中训练模型，需要注意以下几个关键因素：

**数据集**：模型学习的数据来源。它通常包含输入样本及其对应的标签。PyTorch 提供了 torchvision.datasets 模块，方便访问 MNIST、CIFAR-10 和 ImageNet 等常用数据集。用户也可以使用 torch.utils.data.Dataset 类创建自定义数据集。

**数据加载器**：用于在训练期间高效地加载和批处理数据。它负责数据清洗、批处理和并行加载，从而更轻松地以结构化的方式将数据输入模型。这对于性能至关重要，尤其是在处理大型数据集时。

**神经网络模型**：定义了神经网络的结构。我们已经了解到在 PyTorch 中，模型通常是通过继承 torch.nn.Module 并定义网络层和前向传播层来创建的。这包括指定输入和输出维度以及层顺序，例如线性层、激活函数和 dropout。

**损失函数**：衡量模型预测结果与实际目标值之间的差距。它通过提供一个信号来指导优化过程，该信号告诉模型如何调整其参数。常用的损失函数包括用于分类任务的交叉熵损失和用于回归任务的均方误差 (MSE) 损失。我们可以从 torch.nn 中选择预定义的损失函数，也可以定义自己的损失函数。

**优化器**：根据反向传播过程中计算出的梯度更新模型的参数。它决定了模型如何从数据中学习。常用的优化器包括随机梯度下降 (SGD) 和 Adam，它们都可以在 torch.optim 模块中找到。创建优化器时，我们需要指定学习率（一个超参数，用于控制响应梯度而改变参数的程度）和其他超参数。

**训练循环**：实际学习发生的地方。在循环的每次迭代中：

* 从数据加载器 (DataLoader) 获取一批数据。

* 模型执行前向传播以生成预测结果。

* 使用预测结果和真实标签计算损失。

* 通过反向传播计算梯度。

* 优化器根据梯度更新模型参数。

按指定Epoch数量重复此过程，以逐步降低损失并提高模型性能。下面用程序实现模型的训练。

稍等，在训练、测试神经网络前，我们必须先准备好训练和测试数据集。通过以下代码片段，我们可以下载MNIST训练和测试数据集，并将图像转换成一维张量。

In [4]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Training data
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Test data
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Dataloaders
batch_size = 32

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 473kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.34MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


采用下载的训练集训练模型前，我们先选择好损失函数、学习率和优化器。

In [6]:
import torch

learning_rate = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

下面的程序提供训练和评估前向神经网络的方法。

In [7]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (x, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            pred = model(x)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


第一个方法 *train_loop（）* 使用反向传播算法来优化可训练参数，并最小化神经网络的预测误差。第二个方法 *test_loop（）* 使用测试图像计算神经网络的误差，并显示准确率和损失值。

现在，我们可以调用这些方法来训练和评估模型，训练周期设定为 10 个 epoch。

In [8]:
epochs = 10

for t in range(epochs):
    print(f"Epoch {t+1}:")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

Epoch 1:
Accuracy: 92.3%, Avg loss: 1.541904 

Epoch 2:
Accuracy: 94.0%, Avg loss: 1.523087 

Epoch 3:
Accuracy: 94.2%, Avg loss: 1.520421 

Epoch 4:
Accuracy: 94.7%, Avg loss: 1.515731 

Epoch 5:
Accuracy: 95.3%, Avg loss: 1.509619 

Epoch 6:
Accuracy: 95.1%, Avg loss: 1.510913 

Epoch 7:
Accuracy: 95.8%, Avg loss: 1.503975 

Epoch 8:
Accuracy: 95.7%, Avg loss: 1.504209 

Epoch 9:
Accuracy: 95.9%, Avg loss: 1.502651 

Epoch 10:
Accuracy: 96.2%, Avg loss: 1.500809 



模型训练结束后，我们可将训练好的模型结构和参数存储起来，以便测试时调用。

In [9]:
# Set model to evaluation mode
model.eval()

# Trace the model with an example input
traced_model = torch.jit.trace(model, torch.rand(1, 1, 28, 28))

# Save the traced model
traced_model.save("model.pth")

恭喜！我们终于完成了构建神经网络，整理数据集，训练并保存训练好的模型。下面用另一个notebook程序测试我们的模型。